## 1) Importar bibliotecas

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import tensorflow as tf
import polars as pl
import numpy as np

## 2) Ler base de dados

In [ ]:
train_df = pl.read_csv(
    source = "./dados/train_complex.csv"
)

train_df = train_df.with_columns(
    pl.lit(np.random.random(train_df.shape[0])).alias("random_column")
)

train_df = train_df.sort(
    by = "random_column"
).drop("random_column")

print(train_df.shape)
train_df.head(2)

### 2.1) Ver quantidade de classes

In [ ]:
train_df.group_by(
    "class"
).agg(
    pl.col("class").len().alias("total")
)

### 2.2) Ver distribuição das classes

In [ ]:
fig, ax = plt.subplots()

ax.scatter(
    x = train_df["x"],
    y = train_df["y"],
    c = train_df["class"],
    edgecolors = "#000"
)
ax.set_title("Distribuição das classes")

plt.show()

### 2.3) Separar dados

In [ ]:
X = train_df[["x", "y"]].to_numpy()
y = train_df["class"].to_numpy()

print(X[:5])
print(y[:5])

## 3) Criar modelo

In [ ]:
X.shape

### 3.1) Configurar modelo

In [ ]:
neurons = 6

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape = (2, )),
    tf.keras.layers.Dense(units = 2**neurons, activation = "relu"),
    tf.keras.layers.Dense(units = 2**neurons, activation = "relu"),
    tf.keras.layers.Dense(units = 2**neurons, activation = "relu"),
    tf.keras.layers.Dense(units = 2, activation = "softmax"),
])

model_optimizer = tf.keras.optimizers.Adam()
model_loss = tf.keras.losses.SparseCategoricalCrossentropy()
model_metrics = tf.keras.metrics.SparseCategoricalCrossentropy()

model.compile(
    optimizer = model_optimizer,
    loss = model_loss,
    metrics = [model_metrics],
)

model.summary()

### 3.2) Configurar <code>Early_stopping</code>

In [ ]:
model_callback_early_stopping = tf.keras.callbacks.EarlyStopping(
    min_delta = 1E-3,
    patience = 5,
    verbose = 1,
    start_from_epoch = 200,
)

### 3.3) Treinando modelo

In [ ]:
history = model.fit(
    x = X,
    y = y,
    batch_size = 512,
    epochs = 300,
    validation_split = 0.2,
    shuffle = True,
    callbacks = model_callback_early_stopping
)

### 3.4) Avaliando treinamento

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    history.epoch,
    history.history["val_loss"],
    "C0",
    label = "val_loss"
)

ax.plot(
    history.epoch,
    history.history["loss"],
    "C1",
    label = "loss"
)

ax.set_title("Evolução da loss em meio treinamento.")

plt.legend()
plt.show()

### 3.5) Avaliando predição

In [ ]:
X_test = pl.read_csv("./dados/test_complex.csv")

print(X_test.shape)
X_test.head(2)

In [ ]:
fig, axs = plt.subplots(
    ncols = 2,
    figsize = (16, 5)
)

ax1, ax2 = axs[0], axs[1]
predictions = model.predict(X_test[["x", "y"]].to_numpy())
prediction = predictions.argmax(axis = 1)

ax1.scatter(
    x = X_test["x"],
    y = X_test["y"],
    c = X_test["class"],
    edgecolors = "black",
)

ax2.scatter(
    x = X_test["x"],
    y = X_test["y"],
    c = prediction,
    edgecolors = "black"
)

ax1.set_title("Real classification")
ax2.set_title("Prediction by model")

plt.suptitle("Distribution", fontsize = 16)
plt.show()

fig, axs = plt.subplots(
    ncols = 2,
    figsize = (16, 5)
)

axs[0].hist(
    predictions
)

confusion_matrix_plot = ConfusionMatrixDisplay(
    confusion_matrix = confusion_matrix(y_true= X_test["class"], y_pred=prediction),
)

confusion_matrix_plot.plot(
    ax = axs[1],
    
)
axs[0].set_title("Model class priority distribution")
axs[1].set_title("Model performance")

plt.show()